In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
def load_pdfs(file_path: str = "./files") -> list[Document]:
    documents = []
    for pdf_file in Path(file_path).glob("*.pdf"):
        docs = PyPDFLoader(str(pdf_file)).load()
        for doc in docs:
            doc.metadata.update({
                "source": pdf_file.name,
                "file_type": "pdf"
            })
        documents.extend(docs)
    return documents

docs = load_pdfs()

In [ ]:
def split_documents(documents: list[Document], chunk_size: int = 500, chunk_overlap: int = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    return split_docs

In [ ]:
chunks = split_documents(docs)

In [ ]:
import json
import boto3

class BedrockEmbeddingService:
    def __init__(self, region_name: str = "us-east-1", model_id: str = "amazon.titan-embed-text-v2:0"):
        self.client = boto3.client("bedrock-runtime", region_name=region_name)
        self.model_id = model_id

    def embed(self, text: str) -> list[float]:
        response = self.client.invoke_model(
            modelId = self.model_id,
            body = json.dumps({
                "inputText": text
            })
        )
        response_body = json.loads(response['body'].read())
        return response_body.get("embedding", [])

In [ ]:
import os
import httpx

class OpenRouterEmbeddingService:
    def __init__(self, model_id: str = "sentence-transformers/all-minilm-l12-v2"):
        self.model_id = model_id
        self.api_key = os.environ.get("OPENROUTER_API_KEY")
        self.url = "https://openrouter.ai/api/v1/embeddings"

    def embed(self, text: str) -> list[float]:
        print(f"Embedding text using OpenRouter model '{self.model_id}'...")
        response = httpx.post(
            self.url,
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json = {
                "model": self.model_id,
                "input": text
            },
            timeout=30.0
        )
        response.raise_for_status()
        data = response.json()
        print(f"Received embedding of length {len(data['data'][0]['embedding'])}.")
        return data["data"][0]["embedding"]

In [ ]:
from sentence_transformers import SentenceTransformer


class LocalEmbeddingService:

    def __init__(self, model_id: str = "sentence-transformers/all-MiniLM-L12-v2"):
        self.model_id = model_id
        print(f"Loading embedding model '{self.model_id}'...")
        self.model = SentenceTransformer(self.model_id, device="cpu")
        print("Embedding model loaded.")

    def embed(self, text: str) -> list[float]:
        embedding = self.model.encode(
            text,
            normalize_embeddings=True
        )
        return embedding.tolist()

In [ ]:
class S3VectorRepository:
    def __init__(
        self,
        region_name: str = "us-east-1",
        vector_bucket: str = "sample-vector-db",
        vector_index: str = "sample-vector-index-openrouter",
    ):
        self.client = boto3.client("s3vectors", region_name=region_name)
        self.vector_bucket = vector_bucket
        self.index_name = vector_index

    def put_vectors(self, vectors: list[dict]) -> dict:
        print(f"Putting {len(vectors)} vectors into S3 bucket '{self.vector_bucket}' and index '{self.index_name}'...")
        return self.client.put_vectors(
            vectorBucketName=self.vector_bucket,
            indexName=self.index_name,
            vectors=vectors,
        )

    def search(self, embedding: list[float], top_k: int = 5) -> list[dict]:
        response = self.client.query_vectors(
            vectorBucketName=self.vector_bucket,
            indexName=self.index_name,
            queryVector={
                "float32": embedding,
            },
            topK=top_k,
            returnDistance=True,
            returnMetadata=True,
        )
        return response.get("vectors", [])

In [ ]:
class PolicyVectorStore:
    def __init__(self):
        self.embedding_service = LocalEmbeddingService()
        self.vector_repository = S3VectorRepository()

    def add_chunks(self, chunks: list[Document], batch_size: int = 500) -> None:
        vectors = []
        for chunk_number, chunk in enumerate(chunks):
            embedding = self.embedding_service.embed(chunk.page_content)
            source = chunk.metadata.get("source", "unknown")
            page = chunk.metadata.get("page", chunk_number)
            vectors.append({
                "key": f"{source}-{page}-{chunk_number}",
                "data": {
                    "float32": embedding,
                },
                "metadata": {
                    "source_text": chunk.page_content,
                    "document_name": source,
                    "page": page,
                },
            })

            if len(vectors) == batch_size:
                self.vector_repository.put_vectors(vectors)
                vectors = []

        if vectors:
            self.vector_repository.put_vectors(vectors)

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        embedding = self.embedding_service.embed(query)
        return self.vector_repository.search(embedding, top_k=top_k)


vector_store = PolicyVectorStore()

In [ ]:
vector_store.add_chunks(chunks)

In [ ]:
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []
litellm.cache = None

print("LiteLLM callbacks and cache have been reset.")

In [ ]:
import time
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(type="local")

start = time.time()

system_prompt = """
You are a helpful assistant that provides information based on the retrieved documents.
You will be given a user query and a set of retrieved documents. Your task is to generate a concise and accurate response to the user's query using the information from the retrieved documents.
If the information is not available in the retrieved documents, respond with "I'm sorry, I don't have that information."
"""

def chatbot(query: str, top_k: int = 5) -> str:
    retrieved_vectors = vector_store.search(query, top_k=top_k)
    context = "\n\n".join(
        f"Document {index + 1}:\n{vector.get('metadata', {}).get('source_text', '')}"
        for index, vector in enumerate(retrieved_vectors)
    )
    if not context:
        context = "No relevant documents found."

    prompt = f"User Query: {query}\n\nRetrieved Documents:\n{context}\n\nPlease provide a concise and accurate response to the user's query based on the retrieved documents."

    response = completion(
        model="groq/qwen/qwen3.8-27b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        fallbacks=[
            "openrouter/deepseek/deepseek-v4.1-flash",
        ],
    )
    return response.choices[0].message.content

In [ ]:
query = "What are the core principles of Nexora's Employee Code of Conduct?"
response = chatbot(query)

In [ ]:
print(response)